# ZFNet vs CvT-13: the three evaluation scenarios

| Scenario | What it does |
|---|---|
| **S1** | Train both models on the US (public) dataset, test both on the Pakistani (private) dataset **with no Pakistani training**. The label sets differ (196 vs 48 classes), so the test is done at **car-make level** (Toyota, Honda, Suzuki, ...) on the makes that exist in both datasets. |
| **S2** | Train and test both models on the Pakistani dataset with **10-fold stratified cross-validation**. |
| **S3** | Merge both datasets (196 + 48 = 244 classes) and run **10-fold stratified cross-validation** on the combined data. |

**Metrics:** accuracy (top-1), top-5, macro-F1 and macro one-vs-rest AUC (all in %), per fold, then mean and standard deviation.

**Crash recovery:** finished folds are skipped from the CSV, and an interrupted fold continues from its last checkpoint in `OUT_DIR/ckpt/` (saved every `CKPT_EVERY` epochs). After a disconnect: reconnect, mount Drive, re-run all cells from the top with the same `RUN_ID`.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "scikit-learn"], check=False)

import os, time, random, zipfile, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive")
else:
    BASE_DIR = Path(".")

# CONFIG 
US_ZIP  = BASE_DIR / "car_data_us.zip"
PAK_ZIP = BASE_DIR / "car_data_pak.zip"
OUT_DIR = BASE_DIR / "cnn_assignment_outputs"   # reuses zfnet_us.pt / cvt_us.pt if present
OUT_DIR.mkdir(parents=True, exist_ok=True)

FAST    = False
RUN_ID  = "main"
MODELS  = ["zfnet", "cvt"]

SEED, IMG, BATCH, WORKERS = 42, 224, 64, 2
VAL_FRAC  = 0.10
S1_EPOCHS = {"zfnet": 60, "cvt": 15}          # used only if zfnet_us.pt / cvt_us.pt are missing

HP = {
    "zfnet": dict(opt="sgd",   lr=(0.01, 0.01)),
    "cvt":   dict(opt="adamw", lr=(5e-5, 1e-3)),
}
ALIAS = {"daiatsu": "daihatsu", "mercedes-benz": "mercedes", "benz": "mercedes"}



DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("device:", DEVICE)
if DEVICE.type == "cpu":
    print("WARNING: no GPU. Training on CPU will take far too long. Change the runtime type to GPU.")

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

def prepare_dataset(zip_path):
    dest = Path("./data") / zip_path.stem
    if not (dest.exists() and any(dest.iterdir())):
        assert zip_path.exists(), f"{zip_path} not found - check the file name and that Drive is mounted"
        print(f"extracting {zip_path.name} ...")
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as z: z.extractall(dest)
    for d, subdirs, _ in os.walk(dest):
        subdirs[:] = [s for s in subdirs if s != "__MACOSX"]
        if "train" in subdirs and "test" in subdirs: return Path(d)
    raise FileNotFoundError(f"no folder with train/ and test/ inside {zip_path.name}")

US_ROOT, PAK_ROOT = prepare_dataset(US_ZIP), prepare_dataset(PAK_ZIP)
print("US:", US_ROOT, "\nPAK:", PAK_ROOT)

## 1. Data
`train/` and `test/` of each dataset are merged (the cross-validation does the splitting). Every image is available through two views: an augmented one for training and a clean one for validation and testing.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

def merged_views(root):
    aug  = [datasets.ImageFolder(Path(root) / s, train_tf) for s in ("train", "test")]
    ev   = [datasets.ImageFolder(Path(root) / s, eval_tf)  for s in ("train", "test")]
    assert aug[0].classes == aug[1].classes, "train/ and test/ class folders differ"
    targets = np.array(aug[0].targets + aug[1].targets)
    return ConcatDataset(aug), ConcatDataset(ev), targets, aug[0].classes

US_AUG,  US_EVAL,  US_T,  US_CLASSES  = merged_views(US_ROOT)
PAK_AUG, PAK_EVAL, PAK_T, PAK_CLASSES = merged_views(PAK_ROOT)
print(f"US : {len(US_CLASSES)} classes, {len(US_T)} images (min per class {np.bincount(US_T).min()})")
print(f"PAK: {len(PAK_CLASSES)} classes, {len(PAK_T)} images (min per class {np.bincount(PAK_T).min()})")
assert np.bincount(PAK_T).min() >= N_FOLDS and np.bincount(US_T).min() >= N_FOLDS, "a class has fewer images than folds"

# combined dataset for S3: Pakistani labels are shifted by the number of US classes (no classes are merged)
ALL_AUG, ALL_EVAL = ConcatDataset([US_AUG, PAK_AUG]), ConcatDataset([US_EVAL, PAK_EVAL])
ALL_T = np.concatenate([US_T, PAK_T + len(US_CLASSES)])
print(f"COMBINED: {len(US_CLASSES) + len(PAK_CLASSES)} classes, {len(ALL_T)} images")

def make_loaders(ds_aug, ds_eval, tr_idx, va_idx, te_idx):
    kw = dict(batch_size=BATCH, num_workers=WORKERS, pin_memory=USE_AMP, persistent_workers=WORKERS > 0)
    return dict(train=DataLoader(Subset(ds_aug, tr_idx), shuffle=True, drop_last=True, **kw),
                val=DataLoader(Subset(ds_eval, va_idx), shuffle=False, **kw),
                test=DataLoader(Subset(ds_eval, te_idx), shuffle=False, **kw) if te_idx is not None else None)

## 2. Models (same as before)

In [ ]:
class ZFNet(nn.Module):
    def __init__(self, num_classes, bn=True):
        super().__init__()
        def conv(i, o, k, s=1, p=0):
            layers = [nn.Conv2d(i, o, k, s, p, bias=not bn)]
            if bn: layers.append(nn.BatchNorm2d(o))
            layers.append(nn.ReLU(inplace=True)); return layers
        self.features = nn.Sequential(
            *conv(3, 96, 7, 2, 1), nn.MaxPool2d(3, 2), *conv(96, 256, 5, 2), nn.MaxPool2d(3, 2),
            *conv(256, 384, 3, 1, 1), *conv(384, 384, 3, 1, 1), *conv(384, 256, 3, 1, 1), nn.MaxPool2d(3, 2))
        self.pool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256 * 6 * 6, 4096), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(4096, 4096), nn.ReLU(inplace=True), nn.Linear(4096, num_classes))
    def forward(self, x): return self.classifier(torch.flatten(self.pool(self.features(x)), 1))
    def head(self): return self.classifier[-1]

class CvT(nn.Module):
    NAME = "microsoft/cvt-13"
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        from transformers import CvtForImageClassification, CvtConfig
        if pretrained:
            self.m = CvtForImageClassification.from_pretrained(self.NAME, num_labels=num_classes, ignore_mismatched_sizes=True)
        else:
            self.m = CvtForImageClassification(CvtConfig.from_pretrained(self.NAME, num_labels=num_classes))
    def forward(self, x): return self.m(pixel_values=x).logits
    def head(self): return self.m.classifier

def build(name, num_classes, pretrained=True):
    return (ZFNet(num_classes) if name == "zfnet" else CvT(num_classes, pretrained)).to(DEVICE)

## 3. Training and metrics

In [ ]:
def run_epoch(model, loader, criterion, opt=None, scaler=None):
    train = opt is not None
    model.train() if train else model.eval()
    params = [p for p in model.parameters() if p.requires_grad]
    tot_loss = tot_correct = tot = 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
                out = model(x); loss = criterion(out, y)
            if train:
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
                scaler.unscale_(opt); nn.utils.clip_grad_norm_(params, 5.0)
                scaler.step(opt); scaler.update()
            tot_loss += loss.item() * x.size(0); tot_correct += (out.argmax(1) == y).sum().item(); tot += x.size(0)
    return tot_loss / tot, tot_correct / tot

def make_optimizer(model, opt_name, lr_backbone, lr_head):
    head_ids = {id(p) for p in model.head().parameters()}
    head = [p for p in model.parameters() if id(p) in head_ids]
    back = [p for p in model.parameters() if id(p) not in head_ids]
    groups = [dict(params=head, lr=lr_head), dict(params=back, lr=lr_backbone)]
    if opt_name == "sgd": return torch.optim.SGD(groups, momentum=0.9, weight_decay=5e-4, nesterov=True)
    return torch.optim.AdamW(groups, weight_decay=0.05)

CKPT_DIR = OUT_DIR / "ckpt"; CKPT_DIR.mkdir(exist_ok=True)
CKPT_EVERY = 2     # save a resume checkpoint every N epochs (each is ~0.5 GB for ZFNet, ~0.15 GB for CvT)

def fit(model, loaders, epochs, opt_name, lr_backbone, lr_head, tag="", patience=CV_PATIENCE):
    epochs = 1 if FAST else epochs
    opt = make_optimizer(model, opt_name, lr_backbone, lr_head)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_acc, best_state, bad, start, t0 = -1, None, 0, 1, time.time()
    ck = CKPT_DIR / f"{tag}.pt"
    if ck.exists() and not FAST:                      # resume an interrupted run
        try:
            c = torch.load(ck, map_location="cpu")
            model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"])
            sched.load_state_dict(c["sched"]); scaler.load_state_dict(c["scaler"])
            best_acc, best_state, bad, start = c["best_acc"], c["best_state"], c["bad"], c["ep"] + 1
            print(f"[{tag}] resumed from checkpoint, continuing at epoch {start}")
        except Exception as e:
            print(f"[{tag}] checkpoint unreadable ({repr(e)[:80]}), starting from epoch 1")
            best_acc, best_state, bad, start = -1, None, 0, 1
    for ep in range(start, epochs + 1):
        trl, tra = run_epoch(model, loaders["train"], crit, opt, scaler)
        val, vaa = run_epoch(model, loaders["val"], crit)
        sched.step()
        print(f"[{tag}] ep {ep:02d}/{epochs} train {trl:.3f}/{tra:.3f} val {val:.3f}/{vaa:.3f}")
        stop = False
        if vaa > best_acc:
            best_acc, bad = vaa, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: print(f"[{tag}] early stop"); stop = True
        if not FAST and (ep % CKPT_EVERY == 0 or stop) and ep < epochs:
            tmp = ck.with_suffix(".tmp")                # write then rename, so a crash never leaves a half file
            torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                            scaler=scaler.state_dict(), best_acc=best_acc, best_state=best_state, bad=bad, ep=ep), tmp)
            os.replace(tmp, ck)
        if stop: break
    model.load_state_dict(best_state)
    if ck.exists(): ck.unlink()                         # finished: checkpoint no longer needed
    return (time.time() - t0) / 60

@torch.no_grad()
def predict_probs(model, loader):
    model.eval(); P, T = [], []
    for x, y in loader:
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP):
            out = model(x.to(DEVICE, non_blocking=True))
        P.append(torch.softmax(out.float(), 1).cpu()); T.append(y)
    return torch.cat(P).numpy(), torch.cat(T).numpy()

def macro_auc(y, P):
    """Macro one-vs-rest AUC over the classes present in y (needs probabilities P of shape N x K)."""
    aucs = []
    for c in np.unique(y):
        pos = (y == c)
        if pos.all(): continue
        aucs.append(roc_auc_score(pos, P[:, c]))
    return float(np.mean(aucs))

def all_metrics(y, P):
    pred, k = P.argmax(1), min(5, P.shape[1])
    top5 = (np.argsort(-P, axis=1)[:, :k] == y[:, None]).any(1).mean()
    return dict(acc=100 * (pred == y).mean(), top5=100 * top5,
                macro_f1=100 * f1_score(y, pred, average="macro", zero_division=0), auc=100 * macro_auc(y, P))

# results are appended to cv_results_<RUN_ID>.csv after every finished fold
CSV = OUT_DIR / f"cv_results_{RUN_ID}.csv"
ROWS = pd.read_csv(CSV).to_dict("records") if (CSV.exists() and not FAST) else []
DONE = {(r["scenario"], r["model"], int(r["fold"])) for r in ROWS}
print("already finished:", sorted(DONE))

def save_row(row):
    ROWS.append(row); DONE.add((row["scenario"], row["model"], int(row["fold"])))
    if not FAST: pd.DataFrame(ROWS).to_csv(CSV, index=False)
    print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})

## 4. Scenario 1: train on US, test on Pakistani (make level)
The US-trained models (`zfnet_us.pt`, `cvt_us.pt`) are reused if they exist in `OUT_DIR`; otherwise they are trained here on 90% of the US data (10% for validation).

A 196-way model cannot name a Pakistani class, so its 196 probabilities are summed by **car make**. Only makes present in both datasets are scored, the probabilities are renormalised over those makes, and Pakistani images of other makes are dropped. Check the printout below: if a make is spelled differently in the two datasets, add it to `ALIAS` in the config cell.

In [ ]:
def make_of(name):
    s = name.replace("_", " ").replace("-", " ").strip().lower()
    tok = s.split()[0] if s else s
    return ALIAS.get(tok, tok)

US_MAKES  = [make_of(c) for c in US_CLASSES]
PAK_MAKES = [make_of(c) for c in PAK_CLASSES]
SHARED = sorted(set(US_MAKES) & set(PAK_MAKES))
print("US makes  :", sorted(set(US_MAKES)))
print("PAK makes :", sorted(set(PAK_MAKES)))
print("SHARED    :", SHARED)
print("example US classes :", US_CLASSES[:3], "\nexample PAK classes:", PAK_CLASSES[:3])
pak_make_of_img = np.array([SHARED.index(PAK_MAKES[t]) if PAK_MAKES[t] in SHARED else -1 for t in PAK_T])
S1_KEEP = np.where(pak_make_of_img >= 0)[0]
print(f"Pakistani images scored: {len(S1_KEEP)} of {len(PAK_T)} ({len(SHARED)} shared makes)")
print(pd.Series([SHARED[i] for i in pak_make_of_img[S1_KEEP]]).value_counts().to_string())
assert len(SHARED) >= 2, "fewer than 2 shared makes: check the class-name format / ALIAS"

M = np.zeros((len(US_CLASSES), len(SHARED)), dtype=np.float32)
for i, mk in enumerate(US_MAKES):
    if mk in SHARED: M[i, SHARED.index(mk)] = 1.0

def get_us_model(name):
    ck = OUT_DIR / f"{name}_us.pt"
    model = build(name, len(US_CLASSES), pretrained=(not ck.exists() and name == "cvt"))
    if ck.exists():
        model.load_state_dict(torch.load(ck, map_location=DEVICE)); print(f"loaded {ck.name}")
        return model
    set_seed()
    tr, va = train_test_split(np.arange(len(US_T)), test_size=VAL_FRAC, stratify=US_T, random_state=SEED)
    if FAST: tr, va = tr[:300], va[:100]
    fit(model, make_loaders(US_AUG, US_EVAL, tr, va, None), S1_EPOCHS[name], HP[name]["opt"], *HP[name]["lr"], tag=f"{name}-US", patience=8)
    torch.save(model.state_dict(), ck); return model

if RUN_S1:
    keep = S1_KEEP[:200] if FAST else S1_KEEP
    for name in MODELS:
        if ("S1", name, -1) in DONE: print(f"S1 {name} already done"); continue
        model = get_us_model(name)
        loader = DataLoader(Subset(PAK_EVAL, keep), batch_size=BATCH, shuffle=False, num_workers=WORKERS)
        P, _ = predict_probs(model, loader)                       # N x 196
        Pm = P @ M; Pm = Pm / (Pm.sum(1, keepdims=True) + 1e-12)   # N x shared makes
        y = pak_make_of_img[keep]
        row = dict(scenario="S1", model=name, fold=-1, n_train=0, n_test=len(y), minutes=0.0, **all_metrics(y, Pm))
        save_row(row)
        recall = pd.Series({SHARED[c]: 100 * (Pm.argmax(1)[y == c] == c).mean() for c in np.unique(y)})
        print("per-make recall (%):\n", recall.round(1).to_string())
        del model; torch.cuda.empty_cache()

## Notes for the report
- S1 is scored on car **make**, not on exact model, because the two datasets have different model labels. State the shared makes and how many Pakistani images were scored.
- S2 and S3 use 10-fold stratified CV; report mean ± std. In S3 the 196 US and 48 Pakistani classes are kept separate (244 classes).
- CV epochs are reduced (see `CV_EPOCHS`) for compute. Say so.
- The Pakistani images are video frames, so random folds can put near-duplicate frames in train and test; this likely inflates S2 and S3 scores.